# Deep Research 智能体 Cookbook

本 cookbook 演示如何使用 OpenAI Deep Research API 和 OpenAI [Agents SDK](https://openai.github.io/openai-agents-python/) 构建智能体式研究工作流。它是[基础 cookbook](https://cookbook.openai.com/examples/deep_research_api/introduction_to_deep_research_api) 的续篇，如果你还不熟悉那部分内容，建议先阅读。

你将学习如何编排单智能体和多智能体流水线、丰富用户查询以最大化输出质量、流式输出研究进度、集成网络搜索和[用于内部文件搜索的 MCP](https://cookbook.openai.com/examples/deep_research_api/how_to_build_a_deep_research_mcp_server/readme)，以及如何架构一个健壮的研究应用。

对于需要规划、综合、工具使用或多步推理的任务，可以考虑使用 Deep Research 智能体。不要将 Deep Research 用于简单的事实查询、简单问答或简短聊天，这类场景使用普通的 openai.responsesAPI 会更快、更便宜。

### 前置条件
* OpenAI API 密钥（在环境中设置为 OPENAI_API_KEY）
* Agents SDK 和 OpenAI Python SDK

### 准备工作
*安装依赖*

In [ ]:
%pip install --upgrade "openai>=1.88" "openai-agents>=0.0.19"

### 导入库并配置客户端

**零数据保留（Zero Data Retention）**

我们通过下面的 os.environ 设置禁用数据保留。这使企业能够在零数据保留环境中使用 Deep Research。如果数据保留_不是_你当前的约束，可以考虑保持启用，这样你就能为智能体工作流获得自动化的可追溯性，并与评估和微调等其他平台工具深度集成。

In [2]:
import os
from agents import Agent, Runner, WebSearchTool, RunConfig, set_default_openai_client, HostedMCPTool
from typing import List, Dict, Optional
from pydantic import BaseModel
from openai import AsyncOpenAI

# Use env var for API key and set a long timeout
client = AsyncOpenAI(timeout=600.0)
set_default_openai_client(client)
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1" # Disable tracing for Zero Data Retention (ZDR) Organizations

### 基础 Deep Research 智能体

基础研究智能体使用 `gpt-5.6-terra` 模型执行深度研究，该模型在智能和成本之间取得了平衡。它使用 `WebSearchTool` 访问公共互联网，并将研究结果直接流式返回到 notebook 中。

**学习目标：**

完成本节后，你将能够运行单智能体研究任务并流式查看其进度。

In [ ]:
# Define the research agent
research_agent = Agent(
    name="Research Agent",
    model="gpt-5.6-terra",
    tools=[WebSearchTool()],
    instructions="You perform deep empirical research based on the user's question."
)

# Async function to run the research and print streaming progress
async def basic_research(query):
    print(f"Researching: {query}")
    result_stream = Runner.run_streamed(
        research_agent,
        query
    )

    async for ev in result_stream.stream_events():
        if ev.type == "agent_updated_stream_event":
            print(f"\n--- switched to agent: {ev.new_agent.name} ---")
            print(f"\n--- RESEARCHING ---")
        elif (
            ev.type == "raw_response_event"
            and hasattr(ev.data, "item")
            and hasattr(ev.data.item, "action")
        ):
            action = ev.data.item.action or {}
            if action.get("type") == "search":
                print(f"[Web search] query={action.get('query')!r}")

    # streaming is complete → final_output is now populated
    return result_stream.final_output

# Run the research and print the result
result = await basic_research("Research the economic impact of semaglutide on global healthcare systems.")
print(result)

### 带澄清环节的多智能体研究

多智能体深度研究

想一想如何进一步提升 "Deep Research" 产出的研究质量。在这里，我们利用多智能体架构，在将提示词提交给深度研究智能体之前，先用关于用户查询以及我们期望在最终研究报告中看到的内容的_更多信息_来丰富提示词。


## 子智能体提示词增强

这些辅助智能体的提示词经过专门设计，通过为用户的初始查询提供结构和严谨性来提升最终研究输出的质量。

In [5]:
# ─────────────────────────────────────────────────────────────
#  Prompts
# ─────────────────────────────────────────────────────────────

CLARIFYING_AGENT_PROMPT =  """
    If the user hasn't specifically asked for research (unlikely), ask them what research they would like you to do.

        GUIDELINES:
        1. **Be concise while gathering all necessary information** Ask 2–3 clarifying questions to gather more context for research.
        - Make sure to gather all the information needed to carry out the research task in a concise, well-structured manner. Use bullet points or numbered lists if appropriate for clarity. Don't ask for unnecessary information, or information that the user has already provided.
        2. **Maintain a Friendly and Non-Condescending Tone**
        - For example, instead of saying “I need a bit more detail on Y,” say, “Could you share more detail on Y?”
        3. **Adhere to Safety Guidelines**
        """

RESEARCH_INSTRUCTION_AGENT_PROMPT = """

        Based on the following guidelines, take the users query, and rewrite it into detailed research instructions. OUTPUT ONLY THE RESEARCH INSTRUCTIONS, NOTHING ELSE. Transfer to the research agent.

        GUIDELINES:
        1. **Maximize Specificity and Detail**
        - Include all known user preferences and explicitly list key attributes or dimensions to consider.
        - It is of utmost importance that all details from the user are included in the expanded prompt.

        2. **Fill in Unstated But Necessary Dimensions as Open-Ended**
        - If certain attributes are essential for a meaningful output but the user has not provided them, explicitly state that they are open-ended or default to “no specific constraint.”

        3. **Avoid Unwarranted Assumptions**
        - If the user has not provided a particular detail, do not invent one.
        - Instead, state the lack of specification and guide the deep research model to treat it as flexible or accept all possible options.

        4. **Use the First Person**
        - Phrase the request from the perspective of the user.

        5. **Tables**
        - If you determine that including a table will help illustrate, organize, or enhance the information in your deep research output, you must explicitly request that the deep research model provide them.
        Examples:
        - Product Comparison (Consumer): When comparing different smartphone models, request a table listing each model’s features, price, and consumer ratings side-by-side.
        - Project Tracking (Work): When outlining project deliverables, create a table showing tasks, deadlines, responsible team members, and status updates.
        - Budget Planning (Consumer): When creating a personal or household budget, request a table detailing income sources, monthly expenses, and savings goals.
        Competitor Analysis (Work): When evaluating competitor products, request a table with key metrics—such as market share, pricing, and main differentiators.

        6. **Headers and Formatting**
        - You should include the expected output format in the prompt.
        - If the user is asking for content that would be best returned in a structured format (e.g. a report, plan, etc.), ask the Deep Research model to “Format as a report with the appropriate headers and formatting that ensures clarity and structure.”

        7. **Language**
        - If the user input is in a language other than English, tell the model to respond in this language, unless the user query explicitly asks for the response in a different language.

        8. **Sources**
        - If specific sources should be prioritized, specify them in the prompt.
        - Prioritize Internal Knowledge. Only retrieve a single file once.
        - For product and travel research, prefer linking directly to official or primary websites (e.g., official brand sites, manufacturer pages, or reputable e-commerce platforms like Amazon for user reviews) rather than aggregator sites or SEO-heavy blogs.
        - For academic or scientific queries, prefer linking directly to the original paper or official journal publication rather than survey papers or secondary summaries.
        - If the query is in a specific language, prioritize sources published in that language.

        IMPORTANT: Ensure that the complete payload to this function is valid JSON
        IMPORTANT: SPECIFY REQUIRED OUTPUT LANGUAGE IN THE PROMPT
        """

# 四智能体深度研究流水线

1. **分诊智能体（Triage Agent）**  
   - 检查用户的查询  
   - 如果缺少上下文，则路由到澄清智能体；否则路由到指令智能体  

2. **澄清智能体（Clarifier Agent）**  
   - 提出后续问题  
   - 等待用户（或模拟）回答  

3. **指令构建智能体（Instruction Builder Agent）**  
   - 将丰富后的输入转换为精确的研究简报  

4. **研究智能体（Research Agent）**（`gpt-5.6-sol`）  
   - 使用 `WebSearchTool` 执行网络规模的实证研究
   - 使用 MCP 对内部知识库执行搜索，如果有相关文档，智能体会将这些相关片段纳入其参考资料中。   
   - 流式输出中间事件以保证透明度
   - 输出最终的研究产物（我们稍后会对其进行解析）

![../../images/agents_dr.png](../../../images/agent_dr.png)

想更深入地了解 MCP 服务器是_如何_构建的，[请参阅此资源。](https://cookbook.openai.com/examples/deep_research_api/how_to_build_a_deep_research_mcp_server/readme )

In [6]:
# ─────────────────────────────────────────────────────────────
# Structured outputs (needed only for Clarifying agent)
# ─────────────────────────────────────────────────────────────
class Clarifications(BaseModel):
    questions: List[str]

# ─────────────────────────────────────────────────────────────
# Agents
# ─────────────────────────────────────────────────────────────
research_agent = Agent(
    name="Research Agent",
    model="gpt-5.6-sol",
    instructions="Perform deep empirical research based on the user's instructions.",
    tools=[WebSearchTool(),
           HostedMCPTool(
            tool_config={
                "type": "mcp",
                "server_label": "file_search",
                "server_url": "https://<url>/sse",
                "require_approval": "never",
            }
        )
    ]
)

instruction_agent = Agent(
    name="Research Instruction Agent",
    model="gpt-4o-mini",
    instructions=RESEARCH_INSTRUCTION_AGENT_PROMPT,
    handoffs=[research_agent],
)

clarifying_agent = Agent(
    name="Clarifying Questions Agent",
    model="gpt-4o-mini",
    instructions=CLARIFYING_AGENT_PROMPT,
    output_type=Clarifications,
    handoffs=[instruction_agent],
)

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "Decide whether clarifications are required.\n"
        "• If yes → call transfer_to_clarifying_questions_agent\n"
        "• If no  → call transfer_to_research_instruction_agent\n"
        "Return exactly ONE function-call."
    ),
    handoffs=[clarifying_agent, instruction_agent],
)


# ─────────────────────────────────────────────────────────────
#  Auto-clarify helper
# ─────────────────────────────────────────────────────────────
async def basic_research(
    query: str,
    mock_answers: Optional[Dict[str, str]] = None,
    verbose: bool = False,
):
    stream = Runner.run_streamed(
        triage_agent,
        query,
        run_config=RunConfig(tracing_disabled=True),
    )

    async for ev in stream.stream_events():
        if isinstance(getattr(ev, "item", None), Clarifications):
            reply = []
            for q in ev.item.questions:
                ans = (mock_answers or {}).get(q, "No preference.")
                reply.append(f"**{q}**\n{ans}")
            stream.send_user_message("\n\n".join(reply))
            continue
        if verbose:
            print(ev)

    #return stream.final_output
    return stream

# ─────────────────────────────────────────────────────────────
#  Example run
# ─────────────────────────────────────────────────────────────
result = await basic_research(
    "Research the economic impact of semaglutide on global healthcare systems.",
    mock_answers={},   # or provide canned answers
)

## 智能体交互流程

虽然 Agent SDK 的追踪（trace）已原生提供这些信息，但你可能仍希望打印出人类可读的、包含工具调用的高层智能体交互流程。运行 print_agent_interaction 可获得简化且易读的智能体步骤序列，包括：智能体名称、事件类型（交接、工具调用、消息输出）、简要的工具调用信息（工具名称和参数）。


In [ ]:
import json

def parse_agent_interaction_flow(stream):
    print("=== Agent Interaction Flow ===")
    count = 1

    for item in stream.new_items:
        # Agent name, fallback if missing
        agent_name = getattr(item.agent, "name", "Unknown Agent") if hasattr(item, "agent") else "Unknown Agent"

        if item.type == "handoff_call_item":
            func_name = getattr(item.raw_item, "name", "Unknown Function")
            print(f"{count}. [{agent_name}] → Handoff Call: {func_name}")
            count += 1

        elif item.type == "handoff_output_item":
            print(f"{count}. [{agent_name}] → Handoff Output")
            count += 1

        elif item.type == "mcp_list_tools_item":
            print(f"{count}. [{agent_name}] → mcp_list_tools_item")
            count += 1

        elif item.type == "reasoning_item":
            print(f"{count}. [{agent_name}] → Reasoning step")
            count += 1

        elif item.type == "tool_call_item":
            tool_name = getattr(item.raw_item, "name", None)

            # Skip tool call if tool_name is missing or empty
            if not isinstance(tool_name, str) or not tool_name.strip():
                continue  # skip silently

            tool_name = tool_name.strip()

            args = getattr(item.raw_item, "arguments", None)
            args_str = ""

            if args:
                try:
                    parsed_args = json.loads(args)
                    if parsed_args:
                        args_str = json.dumps(parsed_args)
                except Exception:
                    if args.strip() and args.strip() != "{}":
                        args_str = args.strip()

            args_display = f" with args {args_str}" if args_str else ""

            print(f"{count}. [{agent_name}] → Tool Call: {tool_name}{args_display}")
            count += 1

        elif item.type == "message_output_item":
            print(f"{count}. [{agent_name}] → Message Output")
            count += 1

        else:
            print(f"{count}. [{agent_name}] → {item.type}")
            count += 1

# Example usage:
parse_agent_interaction_flow(result)


## 引用

下面是一段 Python 代码，用于提取并打印与最终输出相关的 URL 引用：

In [ ]:
def print_final_output_citations(stream, preceding_chars=50):
    # Iterate over new_items in reverse to find the last message_output_item(s)
    for item in reversed(stream.new_items):
        if item.type == "message_output_item":
            for content in getattr(item.raw_item, 'content', []):
                if not hasattr(content, 'annotations') or not hasattr(content, 'text'):
                    continue
                text = content.text
                for ann in content.annotations:
                    if getattr(ann, 'type', None) == 'url_citation':
                        title = getattr(ann, 'title', '<no title>')
                        url = getattr(ann, 'url', '<no url>')
                        start = getattr(ann, 'start_index', None)
                        end = getattr(ann, 'end_index', None)

                        if start is not None and end is not None and isinstance(text, str):
                            # Calculate preceding snippet start index safely
                            pre_start = max(0, start - preceding_chars)
                            preceding_text = text[pre_start:start].replace('\n', ' ').strip()
                            excerpt = text[start:end].replace('\n', ' ').strip()
                            print("# --------")
                            print("# MCP CITATION SAMPLE:")
                            print(f"#   Title:       {title}")
                            print(f"#   URL:         {url}")
                            print(f"#   Location:    chars {start}–{end}")
                            print(f"#   Preceding:   '{preceding_text}'")
                            print(f"#   Excerpt:     '{excerpt}'\n")
                        else:
                            # fallback if no indices available
                            print(f"- {title}: {url}")
            break

# Usage
print_final_output_citations(result)


In [ ]:
## Deep Research Research Report

print(result.final_output)

### 结论

借助本 notebook 中的模式，你现在已经具备了使用 OpenAI Deep Research 智能体构建可扩展、可用于生产的研究工作流的基础。这些示例不仅演示了如何编排多智能体流水线和流式输出研究进度，还展示了如何集成网络搜索和 MCP 以访问外部知识。

借助智能体式工作流，你可以超越简单问答，处理需要规划、综合和工具使用的复杂多步研究任务。模块化的多智能体设计——分诊、澄清、指令和研究智能体——使你能够将这些流水线适配到广泛的领域和用例，从医疗和金融到技术尽职调查和市场分析。

随着 Deep Research API 和 Agents SDK 的不断演进，这些模式将帮助你始终处于自动化、数据驱动研究的前沿。无论你是在构建内部知识工具、自动化竞争情报，还是为专家分析师提供支持，这些工作流都提供了一个强大且可扩展的起点。

**祝你研究愉快！**
